# HemaCheck - SHAP Interpretasyon ve Explainable AI

**Hedef:** Model kararlarını SHAP değerleri ile yorumlamak.

**Kazanımlar:**
- Hangi özellikler anomali tespitinde kritik?
- Her bir tahminin arkasındaki mantık
- Klinik yorumlanabilirlik

In [ ]:
import sys
sys.path.append('../src')

from preprocessing import load_data, create_features, prepare_features
from models import load_model

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Model ve Veriyi Yükle

In [ ]:
# En iyi modeli yükle
import os
model_files = [f for f in os.listdir('../models') if f.startswith('best_model_') and f.endswith('.pkl')]
print(f"Bulunan modeller: {model_files}")

if model_files:
    model = joblib.load(f'../models/{model_files[0]}')
    print(f"Yüklenen model: {model_files[0]}")
else:
    print("Önce 03_modeling.ipynb çalıştırın!")
    
# Scaler ve feature names
scaler = joblib.load('../models/scaler.pkl')
feature_names = joblib.load('../models/feature_names.pkl')

# Veri hazırla
df = load_data()
df = create_features(df)
X, y, _, _ = prepare_features(df)

# Scale
X_scaled = scaler.transform(X)

## 2. SHAP Explainer Oluştur

In [ ]:
# TreeExplainer kullan (XGBoost/LightGBM/Random Forest için)
explainer = shap.TreeExplainer(model)

# Tüm veri için SHAP değerleri hesapla
print("SHAP değerleri hesaplanıyor...")
shap_values = explainer.shap_values(X_scaled)

print(f"SHAP values shape: {np.array(shap_values).shape}")
print(f"Features: {len(feature_names)}")

## 3. Global Feature Importance

In [ ]:
# Summary plot - Tüm veri seti
plt.figure(figsize=(12, 10))
shap.summary_plot(
    shap_values, 
    X_scaled, 
    feature_names=feature_names,
    show=False,
    max_display=20
)
plt.title('SHAP Feature Importance - Global View', fontsize=14, weight='bold', pad=20)
plt.tight_layout()
plt.savefig('../reports/figures/14_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Bar plot - Mean absolute SHAP values
plt.figure(figsize=(10, 12))
shap.summary_plot(
    shap_values, 
    X_scaled, 
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=20
)
plt.title('Top 20 Features by Mean |SHAP| Value', fontsize=14, weight='bold', pad=20)
plt.tight_layout()
plt.savefig('../reports/figures/15_shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Özellik Etkileşim Analizi

In [ ]:
# En önemli özelliği bul
mean_shap = np.abs(shap_values).mean(axis=0)
top_feature_idx = np.argmax(mean_shap)
top_feature = feature_names[top_feature_idx]

print(f"En önemli özellik: {top_feature}")

# Dependence plot
plt.figure(figsize=(10, 6))
shap.dependence_plot(
    top_feature_idx, 
    shap_values, 
    X_scaled, 
    feature_names=feature_names,
    show=False
)
plt.title(f'SHAP Dependence: {top_feature}', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/16_shap_dependence.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Individual Predictions (Waterfall Plots)

In [ ]:
# Anormal örneği seç ve analiz et
anomaly_indices = np.where(y == 1)[0]
normal_indices = np.where(y == 0)[0]

# İlk anormal örnek
idx_anomaly = anomaly_indices[0]
idx_normal = normal_indices[0]

print(f"Anormal örnek index: {idx_anomaly}")
print(f"Normal örnek index: {idx_normal}")

In [ ]:
# Waterfall plot - Anormal örnek
plt.figure(figsize=(12, 8))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx_anomaly],
        base_values=explainer.expected_value,
        data=X_scaled[idx_anomaly],
        feature_names=feature_names
    ),
    max_display=15,
    show=False
)
plt.title(f'Anomaly Prediction Breakdown (Cell {idx_anomaly})', fontsize=12, weight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/17_shap_waterfall_anomaly.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall plot - Normal örnek
plt.figure(figsize=(12, 8))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx_normal],
        base_values=explainer.expected_value,
        data=X_scaled[idx_normal],
        feature_names=feature_names
    ),
    max_display=15,
    show=False
)
plt.title(f'Normal Prediction Breakdown (Cell {idx_normal})', fontsize=12, weight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/18_shap_waterfall_normal.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Force Plot - Interaktif Yorumlama

In [ ]:
# Force plot - Anormal örnek
shap.force_plot(
    explainer.expected_value, 
    shap_values[idx_anomaly], 
    X_scaled[idx_anomaly],
    feature_names=feature_names,
    matplotlib=True,
    show=False
)
plt.title(f'Force Plot: Anormal Hücre (index {idx_anomaly})')
plt.tight_layout()
plt.savefig('../reports/figures/19_shap_force_anomaly.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Top Özellikler Analizi

In [ ]:
# Top 10 özellik ve önem dereceleri
top_n = 10
mean_shap_values = np.abs(shap_values).mean(axis=0)
top_indices = np.argsort(mean_shap_values)[-top_n:][::-1]

print(f"\n=== TOP {top_n} ÖNEMLİ ÖZELLİKLER ===")
for i, idx in enumerate(top_indices, 1):
    print(f"{i:2d}. {feature_names[idx]:30s} | SHAP: {mean_shap_values[idx]:.4f}")
    
# Özellik gruplarına göre analiz
morphological_keywords = ['diameter', 'nucleus', 'chromatin', 'cytoplasm', 'circularity', 
                         'eccentricity', 'granularity', 'lobularity', 'membrane', 'shape']
color_keywords = ['mean_r', 'mean_g', 'mean_b', 'stain', 'color']
clinical_keywords = ['wbc', 'rbc', 'hemoglobin', 'hematocrit', 'platelet', 'mcv', 'mchc']

morph_importance = sum(mean_shap_values[i] for i, name in enumerate(feature_names) 
                      if any(kw in name.lower() for kw in morphological_keywords))
color_importance = sum(mean_shap_values[i] for i, name in enumerate(feature_names) 
                      if any(kw in name.lower() for kw in color_keywords))
clinical_importance = sum(mean_shap_values[i] for i, name in enumerate(feature_names) 
                         if any(kw in name.lower() for kw in clinical_keywords))
other_importance = sum(mean_shap_values) - morph_importance - color_importance - clinical_importance

print(f"\n=== ÖZELLİK GRUP ÖNEMİ ===")
print(f"Morfolojik: {morph_importance:.4f} ({100*morph_importance/sum(mean_shap_values):.1f}%)")
print(f"Renk:       {color_importance:.4f} ({100*color_importance/sum(mean_shap_values):.1f}%)")
print(f"Klinik:     {clinical_importance:.4f} ({100*clinical_importance/sum(mean_shap_values):.1f}%)")
print(f"Diğer:      {other_importance:.4f} ({100*other_importance/sum(mean_shap_values):.1f}%)")

In [ ]:
# Grup önemini görselleştir
fig, ax = plt.subplots(figsize=(10, 6))

groups = ['Morfolojik', 'Renk', 'Klinik', 'Diğer']
values = [morph_importance, color_importance, clinical_importance, other_importance]
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

bars = ax.bar(groups, values, color=colors, edgecolor='black')
ax.set_ylabel('Toplam |SHAP| Değeri', fontsize=12)
ax.set_title('Özellik Grubu Önem Dağılımı', fontsize=14, weight='bold')

for bar, val in zip(bars, values):
    pct = 100 * val / sum(values)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
           f'{pct:.1f}%', ha='center', fontsize=11, weight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/20_feature_groups.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Klinik Yorumlama

In [ ]:
print("="*60)
print("       SHAP ANALİZİ - KLİNİK YORUMLAMA")
print("="*60)

print("\n🔬 ANOMALİ TESPİTİNDE KRİTİK FAKTÖRLER:")
print("\n1. MORFOLOJİK ÖZELLİKLER (%{:.1f} önem):".format(100*morph_importance/sum(mean_shap_values)))
print("   • Hücre çapı ve şekli (circularity, eccentricity)")
print("   • Çekirdek/cytoplasm oranı")
print("   • Granularity ve lobularity skorları")

print("\n2. RENK ÖZELLİKLERİ (%{:.1f} önem):".format(100*color_importance/sum(mean_shap_values)))
print("   • RGB ortalamaları (boyama yoğunluğu)")
print("   • Stain intensity (Giemsa/Wright boyama)")

print("\n3. KAN SAYIMI VERİLERİ (%{:.1f} önem):".format(100*clinical_importance/sum(mean_shap_values)))
print("   • WBC, RBC, Platelet sayıları")
print("   • Hemoglobin ve hematokrit")

print("\n" + "="*60)
print("NOT: Bu model kan hücresi morfolojisinden anomali")
print("      tespiti yapar. Klinik kullanımda patolog")
print("      onayı gereklidir.")
print("="*60)

## 9. Özet

**SHAP Analizi Bulguları:**

1. **En Önemli Özellikler:**
   - `nucleus_area_pct` - Çekirdek alanı yüzdesi
   - `chromatin_density` - Kromatin yoğunluğu
   - `cell_diameter_um` - Hücre çapı
   - `stain_intensity` - Boyama yoğunluğu

2. **Özellik Grupları:**
   - Morfolojik: ~50%
   - Renk: ~25%
   - Klinik: ~15%

3. **Kaydedilen Görselleştirmeler:**
   - `14_shap_summary.png` - Global SHAP özeti
   - `15_shap_bar.png` - Feature importance bar
   - `16_shap_dependence.png` - Dependence plot
   - `17_shap_waterfall_anomaly.png` - Anomali waterfall
   - `18_shap_waterfall_normal.png` - Normal waterfall
   - `19_shap_force.png` - Force plot
   - `20_feature_groups.png` - Grup önemi

**Proje Tamamlandı!** GitHub ve LinkedIn içerikleri için `reports/` klasörüne bakın.